# Day 1 / Session 2 -- Prompt Engineering Demos

Six live demos that show how prompt design controls LLM behavior.
Run every cell top-to-bottom; each demo builds your intuition for
the patterns you will apply when you build your own agent components later.

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn langchain langchain-openai langchain-core python-dotenv

## Environment Setup

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import json

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

---
## Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.
**Few-shot** prompting provides a handful of examples so the model can learn
the expected format and behavior.

We compare both approaches on a **client feedback classification** task --
categorizing engagement survey responses as positive, negative, or neutral.

In [ ]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Classify the sentiment of this client engagement feedback as positive, negative, or neutral: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples from past engagement surveys
few_shot_messages = [
    {"role": "system", "content": "You are a client feedback classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Feedback: 'The engagement exceeded our expectations \u2014 the team was exceptional and the insights transformed our strategy.'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Feedback: 'Disappointed with the engagement. Deliverables were late and the analysis did not address our core issues.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Feedback: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}
]
few_shot_response = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

---
## Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning
process step by step, which improves accuracy on math, logic, and multi-step
problems.

This mirrors structured problem-solving -- breaking a problem into steps
before jumping to the answer. CoT is especially valuable for market sizing
and financial analysis.

In [ ]:
# Demo 2: Chain-of-Thought Prompting

problem = "A McKinsey client has 1,200 retail stores. They plan to close 20% of underperforming locations and increase revenue per remaining store by 15%. If current average revenue per store is $2M, what will total revenue be after the restructuring?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT -- mirroring structured problem-solving
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many stores will be closed\n2. Then, find the number of remaining stores\n3. Calculate the new revenue per store after the 15% increase\n4. Compute total projected revenue"
response_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

---
## Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different practice-area
personas that approach the same client question from different angles. This is
foundational for building specialized agents in multi-agent systems.

In [5]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "McKinsey Growth Strategy Lead": "You are a McKinsey partner in Growth & Innovation. You approach every question by identifying growth levers, market opportunities, and revenue acceleration strategies. Use frameworks like the Three Horizons of Growth.",
    "McKinsey Risk & Compliance Expert": "You are a McKinsey senior expert in Risk & Resilience. You evaluate everything through the lens of regulatory risk, compliance requirements, operational resilience, and enterprise risk management.",
    "McKinsey Organization & People Lead": "You are a McKinsey partner in People & Organizational Performance. You think about talent strategy, organizational design, change management, and leadership development. Focus on the human side of transformation."
}
question = "Our banking client is considering launching a digital-only subsidiary to compete with fintechs."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")


McKinsey Growth Strategy Lead:
To assist your banking client in launching a digital-only subsidiary, we should systematically explore growth levers, market opportunities, and revenue acceleration strategies using the frameworks such as the Three Horizons of Growth. Here’s a strategic approach tailored for this scenario:

### 1. **Assessing Market Opportunities**

#### a. **Market Landscape Analysis**
- **Competitor Analysis**: Identify key fintech competitors and assess their strengths, weaknesses, product offerings, and customer segments.
- **Customer Segmentation**: Define target customer segments based on demographics, tech savviness, financial behavior, and pain points that can be addressed by a digital-only bank.
- **Regulatory Considerations**: Evaluate the regulatory landscape relevant to digital banking in the intended markets for compliance and strategic advantages.
  
#### b. **Market Trends**
- **Digital Banking Growth**: Leverage trends in digital banking adoption, consume

**Observation:** Each persona focuses on different aspects of the same question -- growth levers, regulatory risk, or talent and org design. This demonstrates how system prompts shape agent behavior.

---
## Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems, we often need the LLM output to be
machine-readable -- for example, extracting structured client data from
meeting notes or parsing engagement details from unstructured text.
JSON mode ensures the response is valid JSON, making downstream processing
reliable.

In [6]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "Sarah Chen is the CFO of Meridian Healthcare, a $3.2B revenue hospital network based in Chicago. She has 18 years of experience in healthcare finance and previously led M&A at Deloitte. Her priorities for the McKinsey engagement are cost optimization, revenue cycle management, and evaluating two potential acquisition targets."

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "Extract the client executive's information and return it as a JSON object with keys: name, title, company, company_revenue, location, experience_years, previous_employer, engagement_priorities (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nClient Executive: {parsed.get('name')}, {parsed.get('title')}")
print(f"Engagement Priorities: {', '.join(parsed.get('engagement_priorities', []))}")

Parsed JSON output:
{
  "name": "Sarah Chen",
  "title": "CFO",
  "company": "Meridian Healthcare",
  "company_revenue": "$3.2B",
  "location": "Chicago",
  "experience_years": 18,
  "previous_employer": "Deloitte",
  "engagement_priorities": [
    "cost optimization",
    "revenue cycle management",
    "evaluating two potential acquisition targets"
  ]
}

Client Executive: Sarah Chen, CFO
Engagement Priorities: cost optimization, revenue cycle management, evaluating two potential acquisition targets


**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

---
## Demo 5: Multi-Turn Conversation Management

Agents need to maintain context across multiple exchanges -- remembering the
client industry, engagement scope, and prior recommendations. This demo shows
how to build a conversation manager that accumulates context, mirroring how
an engagement progresses through multiple discussions.

In [7]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo: Multi-turn engagement planning conversation
conv = ConversationManager("You are a McKinsey engagement planning assistant. Help consultants scope and plan client engagements. Be structured, use MECE principles, and provide actionable next steps.")

print("User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.")
print("Assistant:", conv.send("We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%."))

print("\n" + "-"*50)
print("\nUser: What data should we request from the client in the first week?")
print("Assistant:", conv.send("What data should we request from the client in the first week?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize our engagement scope so far?")
print("Assistant:", conv.send("Can you summarize our engagement scope so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.
Assistant: Congratulations on winning the new engagement! To effectively scope and plan this project with the goal of reducing claims processing costs by 30%, we will outline the engagement using the MECE framework. 

### Engagement Scope and Structure

**1. Understanding the Current State**
   - **A. Claims Processing Analysis**
     - Map the current claims process (end-to-end).
     - Identify the key stakeholders involved (e.g., claims handlers, IT, customer service).
     - Analyze historical claims processing costs (fixed vs. variable components).

   - **B. Performance Metrics**
     - Evaluate current KPIs (e.g., processing time, accuracy rate, customer satisfaction).
     - Benchmark against industry standards and best practices.

**2. Cost Breakdown**
   - **A. Direct Costs**
     - Labor costs (salaries, overtime) associated with claims processing.
     -

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing insurance claims without being reminded. The conversation history grows with each exchange.

---
## Demo 6: Structured Outputs with LangChain Prompt Templates

LangChain provides a higher-level abstraction for prompt engineering. Its
`ChatPromptTemplate` supports parameterized prompts, and `OutputParser`
classes enforce structured output formats (JSON, lists, etc.) -- reducing
boilerplate and improving reliability.

In [8]:
# Demo 6: Structured Outputs with LangChain Prompt Templates

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Step 1: Initialize the LangChain LLM wrapper
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# --- Part A: ChatPromptTemplate with variables ---
print("PART A: LangChain ChatPromptTemplate for Analysis")
print("=" * 60)

# Reusable template for industry analysis across different client engagements
analysis_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey {practice_area} expert. Provide structured, partner-ready analysis."),
    ("user", "Analyze the following for our client engagement: {topic}\n\nProvide your analysis in exactly 3 bullet points with strategic implications.")
])

prompt_value = analysis_template.invoke({"practice_area": "Operations", "topic": "supply chain resilience in the semiconductor industry"})
response = llm.invoke(prompt_value)
print(f"Practice: Operations | Topic: semiconductor supply chain")
print(response.content)

print()
prompt_value2 = analysis_template.invoke({"practice_area": "Strategy & Corporate Finance", "topic": "consolidation trends in European banking"})
response2 = llm.invoke(prompt_value2)
print(f"Practice: Strategy | Topic: European banking consolidation")
print(response2.content)

# --- Part B: Pydantic-based Structured Output for deliverables ---
print("\n" + "=" * 60)
print("PART B: Pydantic Structured Output -- Engagement Summary")
print("=" * 60)

class EngagementSummary(BaseModel):
    executive_summary: str = Field(description="A 1-2 sentence executive summary suitable for a CEO briefing")
    key_findings: list[str] = Field(description="A list of 3 key findings")
    strategic_priority: str = Field(description="The single most important strategic priority: growth, cost_optimization, digital_transformation, or M&A")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")

parser = JsonOutputParser(pydantic_object=EngagementSummary)
format_instructions = parser.get_format_instructions()
print(f"Format instructions (auto-generated):\n{format_instructions}\n")

structured_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey engagement manager preparing a structured deliverable."),
    ("user", "Analyze the strategic landscape for: {topic}\n\n{format_instructions}")
])

prompt = structured_template.invoke({
    "topic": "a mid-size US retailer considering an omnichannel transformation",
    "format_instructions": format_instructions
})

response = llm.invoke(prompt)
parsed_output = parser.parse(response.content)

print("Parsed structured output:")
for key, value in parsed_output.items():
    print(f"  {key}: {value}")

# --- Part C: LangChain Chain (Prompt | LLM | Parser) ---
print("\n" + "=" * 60)
print("PART C: LangChain Chain -- Prompt | LLM | Parser")
print("=" * 60)

chain = structured_template | llm | parser

result = chain.invoke({
    "topic": "a healthcare provider evaluating AI-powered diagnostics for radiology",
    "format_instructions": format_instructions
})

print("Chain output (directly parsed):")
print(json.dumps(result, indent=2))

PART A: LangChain ChatPromptTemplate for Analysis
Practice: Operations | Topic: semiconductor supply chain
1. **Supply Chain Vulnerabilities and Geopolitical Risks**: The semiconductor industry is highly susceptible to geopolitical tensions, particularly between major players like the U.S. and China. Recent disruptions, such as export controls and trade restrictions, have highlighted the fragility of global supply chains. Strategic Implication: Companies should diversify their supply sources and consider regionalizing production to mitigate risks associated with geopolitical instability, ensuring a more resilient supply chain.

2. **Technological Advancements and Capacity Constraints**: The rapid pace of technological innovation in semiconductor manufacturing, coupled with capacity constraints, has led to significant supply-demand imbalances. The COVID-19 pandemic exacerbated these issues, resulting in prolonged lead times and shortages. Strategic Implication: Firms should invest in ad

**Observation:** LangChain templates let you swap variables (practice area, topic) without rewriting the prompt each time. The Pydantic parser guarantees the output matches your schema, and the chain operator (`|`) composes prompt, model, and parser into a single callable pipeline.

---
## Key Takeaways

| Demo | Pattern | Why It Matters |
| --- | --- | --- |
| 1 | Zero-shot vs. Few-shot | Few-shot examples lock down output format |
| 2 | Chain-of-Thought | Step-by-step reasoning improves accuracy on quantitative tasks |
| 3 | Role-based personas | System prompts define agent specialization |
| 4 | JSON mode | Machine-readable output for downstream pipelines |
| 5 | Multi-turn management | Context accumulation enables stateful agents |
| 6 | LangChain templates | Reusable, composable prompt + parser chains |

Next up: **Exercises** -- you will build a research-agent system prompt yourself.